# Smart-G demo notebook for Objects

This is an interactive document allowing to run Smart-G with python and visualize the results. <br>
*Tips*: cells can be executed with shift-enter. Tooltips can be obtained with shift-tab. More information [here](http://ipython.org/notebook.html) or in the help menu. [A table of content can also be added](https://github.com/minrk/ipython_extensions#table-of-contents).

In [ ]:
%matplotlib inline
# next 2 lines allow to automatically reload modules that have been changed externally
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

try:
    import subprocess
    check = subprocess.check_call(['git', 'rev-parse', '--show-toplevel'],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    ROOTPATH = subprocess.Popen(['git', 'rev-parse', '--show-toplevel'],
                                stdout=subprocess.PIPE).communicate()[0].rstrip().decode('utf-8') # Root Git Path
    ROOTPATH = Path(ROOTPATH)
except subprocess.CalledProcessError:
    ROOTPATH = Path.cwd()
sys.path.insert(0, str(ROOTPATH))

from smartg.config import DIR_AUXDATA
from smartg.smartg import Smartg, LambSurface, CusForward, CusBackward, Sensor
from smartg.atmosphere import AtmAFGL, AerOPAC
from smartg.albedo import Albedo_cst
from smartg.visualizegeo import Mirror, Plane, Transformation, \
    Entity, Matte, generateHfP, generateHfA, extractPoints, Heliostat, visualize_entity
from smartg.tools.smartg_view import smartg_view, receiver_view, cat_view, nopt_view
from luts.luts import Idx
import numpy as np

import warnings
warnings.simplefilter('always', DeprecationWarning)
import geoclide as gc

In [ ]:
# Uncomment below for 3D views
%matplotlib widget

# Quick Start

## Creation of objects (simple example)

In [ ]:
# We want to create a simple case with a receiver and four heliostats (MirA, B, C and D).
# The receiver will be created a x = 1km, the first heliostat at x = 950m, the
# second heliostat at x = 900m and so on. The mirror will be oriented such that the
# direct rays reflected by the mirrors will totally reach the receiver.

# The sun zenith angle
SZA = 14.3
# Create the heliostats MirA, MirB, MirC and MirD (smartg unit is the kilometer)
wMx = 0.004725
wMy = 0.00642
MirA = Entity(name = "reflector", \
              materialAV = Mirror(reflectivity = 0.88), \
              materialAR = Matte(), \
              geo = Plane( p1 = gc.Point(-wMx, -wMy, 0.),
                           p2 = gc.Point(wMx, -wMy, 0.),
                           p3 = gc.Point(-wMx, wMy, 0.),
                           p4 = gc.Point(wMx, wMy, 0.) ), \
              transformation = Transformation( rotation = np.array([0., 20.281725, 0.]), \
                                               translation = np.array([-0.05, 0., 0.00517]) ))

# MirB, C and D are exact copies of MirA, but with different transformations
MirB = Entity(MirA)
MirC = Entity(MirA)
MirD = Entity(MirA)
MirB.set_transformation(Transformation( rotation = np.array([0., 29.460753, 0.]), \
                                        translation = np.array([-0.1, 0., 0.00517]) ))
MirC.set_transformation(Transformation( rotation = np.array([0., 35.129831, 0.]), \
                                        translation = np.array([-0.15, 0., 0.00517]) ))
MirD.set_transformation(Transformation( rotation = np.array([0., 38.715473, 0.]), \
                                        translation = np.array([-0.2, 0., 0.00517]) ))

# Create the receiver Recept1
wRx = 0.006
wRy = 0.007
# TC = Taille cellule. The receiver is devided in several cells to visualize the flux distribution 
Rec1 = Entity(name = "receiver", TC = 0.0005, \
                 materialAV = Matte(reflectivity = 0.), \
                 materialAR = Matte(reflectivity = 0.), \
                 geo = Plane( p1 = gc.Point(-wRx, -wRy, 0.),
                              p2 = gc.Point(wRx, -wRy, 0.),
                              p3 = gc.Point(-wRx, wRy, 0.),
                              p4 = gc.Point(wRx, wRy, 0.) ), \
                 transformation = Transformation( rotation = np.array([0., -101.5, 0.]), \
                                                  translation = np.array([0., 0., 0.1065]) ))

# Create the list containing all the objects
LOBJ1 = [MirA, MirB, MirC, MirD, Rec1]

## How to verify the drawing of objects

In [ ]:
# 3D print of objects (the notebook give only a 2D print of the 3D one)
fig1 = visualize_entity(entities=[MirA, MirB, MirC, MirD, Rec1], th_deg=SZA, 
                        show_rays=True, sr_view=1, rs_fac=1)

## Run the simulation

In [ ]:
# In this simulation the photons from TOA are launched to fill only the
# mirrors (ray tracing method or restricted forward method) --> cusForward = CusForward(LMODE="RF")
# By default the sun is a ponctual sun source targeting the origin (0,0,0) --> --> cusForward = None
# For a full forward mode i.g. specify in Smartg.run() -> cusForward = CusForward(CFX=10, CFY=10, LMODE="FF")
# Where CFX is the size in kilometer in the x axis of the rectangle. Be careful, the full forward mode 
# need a big number of photons to obtain a good accuracy !

w2 = 0.5
Pmin = [-w2, -w2, -0.005]
Pmax = [w2, w2, 0.125]
interval0 = [Pmin, Pmax] # enable to earn some computational time, can be very usefull in FF mode
aer = AerOPAC('desert', 0.25, 550.)
pro = AtmAFGL('afglms', comp=[aer], P0 = 877, H2O=1.2)
m = Smartg(double = True, obj3D = True).run( surf = LambSurface(ALB=Albedo_cst(0.25)), 
    THVDEG=SZA, NF=1e6, wl=550., NBPHOTONS=4e7, NBLOOP = 5e6, atm=pro,
    myObjects=LOBJ1, interval = interval0, cusL = CusForward(LMODE="RF"))
m_ds = m.to_xarray()


## How to show the results ?

In [ ]:
# Show the description of the simulation
m.print_info(show_attrs=True)

In [ ]:
# print the infomation at the receiver
output = cat_view(m_ds)
print("\noutput:")
output

In [ ]:
# print the infomation at the receiver
output = cat_view(m_ds)
print("\noutput:")
output

In [ ]:
# print the total flux distribution at the receiver
receiver_view(ds_sg_out=m_ds)

In [ ]:
# print the flux distribution at the receiver from only the direct D (cat1)
receiver_view(ds_sg_out=m_ds, cat=1)

In [ ]:
# print the flux distribution at the receiver from beams from the process H only (cat2)
receiver_view(ds_sg_out=m_ds, cat=2)

In [ ]:
# print the flux distribution at the receiver from beams from the process E only (cat3)
receiver_view(ds_sg_out=m_ds, cat=3)

In [ ]:
# print the flux distribution at the receiver from beams from the process A only (cat4)
receiver_view(ds_sg_out=m_ds, cat=4)

In [ ]:
# print the flux distribution at the receiver from beams from the two processes H and A (cat5)
receiver_view(ds_sg_out=m_ds, cat=5)

In [ ]:
# print the flux distribution at the receiver from beams from the two processes H and E (cat6)
receiver_view(ds_sg_out=m_ds, cat=6)

In [ ]:
# print the flux distribution at the receiver from beams from the two processes E and A (cat7)
receiver_view(ds_sg_out=m_ds, cat=7)

In [ ]:
# print the flux distribution at the receiver from beams from the three processes H , E and A (cat8)
receiver_view(ds_sg_out=m_ds, cat=8)

# Complex cases

## Quick heliostat generation by giving two angles

In [ ]:
# Specify the sun zenith angle
SZA= 14.3

# Creation of a receiver
wRx = 0.006
wRy = 0.007

Rec2 = Entity(name = "receiver", TC = 0.0005, \
                 materialAV = Matte(reflectivity = 0.), \
                 materialAR = Matte(reflectivity = 0.), \
                 geo = Plane( p1 = gc.Point(-wRx, -wRy, 0.),
                              p2 = gc.Point(wRx, -wRy, 0.),
                              p3 = gc.Point(-wRx, wRy, 0.),
                              p4 = gc.Point(wRx, wRy, 0.) ), \
                 transformation = Transformation( rotation = np.array([0., -101.5, 0.]), \
                                                  translation = np.array([0., 0., 0.1065]) ))

pRe = gc.Point(Rec2.transformation.transx, Rec2.transformation.transy, Rec2.transformation.transz)

# Generation of heliostats thanks to two angles, MINANG and MAXANG.
LOBJ2 = generateHfA(THEDEG=SZA, PR=pRe, MINANG=150, MAXANG=210, GAPDEG = 10,
                        FDRH=0.1, NBH = 3, GAPDIST = 0.008, HSX = 0.005, HSY = 0.01,
                        PILLH= 0.00517, REF=0.88)

# Without forgetting to add the reveicer at the list of objects
LOBJ2.append(Rec2)

### Drawing verification

In [ ]:
fig1 = visualize_entity(LOBJ2, th_deg=SZA)

### Run the simulation

In [ ]:
aer = AerOPAC('desert', 1, 550.)
pro = AtmAFGL('afglms', comp=[aer])
lMode1 = CusForward(LMODE="RF")

m2 = Smartg(double = True, obj3D = True).run( surf = LambSurface(ALB=Albedo_cst(0.25)), 
        THVDEG=SZA, NF=1e6, wl=550., NBPHOTONS=1e7, NBLOOP = 1e6, atm=pro,
        myObjects=LOBJ2, cusL = lMode1)
m2_ds = m2.to_xarray()


In [ ]:
m2.print_info(show_attrs=True)

### Results

In [ ]:
cat_view(m2_ds, accuracy=3)

In [ ]:
receiver_view(ds_sg_out=m2_ds, log_color_scale=True)

## Quick heliostat generation by giving positions from a file

In [ ]:
# You need a file with the x, y and z positions, see the file HPOS_STP1.dat as example
# Points of heliostats are extracted from the given file as a list of class point
lPH = extractPoints(filename = DIR_AUXDATA / 'STPs' / 'STP1.dat')

# Specify the solar zenith angle
SZA = 14.3

# Creation of a receiver
wRx = 0.006
wRy = 0.007

Rec3 = Entity(name = "receiver", TC = 0.0005, \
                 materialAV = Matte(reflectivity = 0.), \
                 materialAR = Matte(reflectivity = 0.), \
                 geo = Plane( p1 = gc.Point(-wRx, -wRy, 0.),
                              p2 = gc.Point(wRx, -wRy, 0.),
                              p3 = gc.Point(-wRx, wRy, 0.),
                              p4 = gc.Point(wRx, wRy, 0.) ), \
                 transformation = Transformation( rotation = np.array([0., -101.5, 0.]), \
                                                  translation = np.array([0., 0., 0.1065]) ))

# Coordinate of the center of the created receiver needed for heliostat generation
pRe = gc.Point(Rec3.transformation.transx, Rec3.transformation.transy, Rec3.transformation.transz)

# Generate heliostats thanks to a list of Points, lPH.
LOBJ3 = generateHfP(THEDEG = SZA, PH = lPH, PR = pRe,
                        HSX = 0.00945, HSY = 0.01284, REF = 0.88)

# Without forgetting to add the reveicer at the list of objects
LOBJ3.append(Rec3)

### Drawing verification

In [ ]:
fig1 = visualize_entity(LOBJ3, th_deg=SZA)

### Run the simulation

In [ ]:
Pmin = [-0.12, -0.05, -0.05]
Pmax = [0.05, 0.05, 0.125]
interval0 = [Pmin, Pmax] # enable to earn some computational time, can be very usefull in FF mode
aer = AerOPAC('desert', 0.2, 550.)
pro = AtmAFGL('afglms', comp=[aer])
lMode = CusForward(CFX = 0.06, CFY = 0.08, CFTX = -0.08, CFTY = 0., LMODE="FF")
m3 = Smartg(double = True, obj3D = True).run( surf = LambSurface(Albedo_cst(0.25)), 
        THVDEG=SZA, NF=1e6, wl=550., NBPHOTONS=4e7, NBLOOP = 2e6, atm=pro,
        myObjects=LOBJ3, interval = interval0, cusL = lMode)
m3_ds = m3.to_xarray()


In [ ]:
m3.print_info(show_attrs=True)

### Results

In [ ]:
cat_view(m3_ds)

In [ ]:
receiver_view(ds_sg_out=m3_ds, vmin=0)

# More complex cases

## Deals with 3D object in backward mode

### Creation of the objects

In [ ]:
# Zenith and Azimuth angle of the sun, respectively Theta and phi
SZA = 50
SAA = 45
PHI = 180.-SAA

# Position of heliostats (list) [pH] and position of the sensor pS
pH = [gc.Point(-0.05, 0., 0.00517)] # here only one
pS = gc.Point(0., 0., 0.1065)

# create the heliostats : here we need the zenith and azimuth angles
LOBJ4 = generateHfP(THEDEG = SZA, PHIDEG = PHI, PH = pH, PR = pS, 
                    HSX = 0.00945, HSY = 0.01284, REF = 0.88)

# Modify the rugosity ? here -->
LOBJ4[0].materialAV = Mirror(reflectivity = 0.88, roughness=0.1, shadow=True)

# Creation of the sensor
vS = gc.Vector(pH[0]-pS) # direction of the sensor described by a vector
SEN = Sensor(POSX = pS.x, POSY = pS.y, POSZ = pS.z, LOC  = 'ATMOS', FOV  = 0., TYPE = 0, V=vS)

### Run the simulation

In [ ]:
aer = AerOPAC('desert', 0.2, 550.)
pro = AtmAFGL('afglms', comp=[aer]).calc(550.)

m4 = Smartg(double = True, obj3D = True, back = True).run( surf = LambSurface(ALB=Albedo_cst(0.25)),
                NF=1e6, wl=550., NBPHOTONS=1e8, NBLOOP = 1e7, atm=pro,
                myObjects=LOBJ4, sensor = SEN)

### See the results

In [ ]:
_=smartg_view(m4, QU=True, ind=Idx([0, 22, 44, 68, 90, 112, 134, 156]))

## STP construction with heliostats composed of facets

In [ ]:
# Specify the sun zenith angle and sun azimuth angle
SZA = 50
SAA = 100
PHI = 180. - SAA

# Creation of a receiver
wRx = 0.006
wRy = 0.007

Rec5 = Entity(name = "receiver", TC = 0.0005, \
                 materialAV = Matte(reflectivity = 0.), \
                 materialAR = Matte(reflectivity = 0.), \
                 geo = Plane( p1 = gc.Point(-wRx, -wRy, 0.),
                              p2 = gc.Point(wRx, -wRy, 0.),
                              p3 = gc.Point(-wRx, wRy, 0.),
                              p4 = gc.Point(wRx, wRy, 0.) ), \
                 transformation = Transformation( rotation = np.array([0., -101.5, 0.]), \
                                                  translation = np.array([0., 0., 0.1065]) ))

pRe = gc.Point(Rec5.transformation.transx, Rec5.transformation.transy, Rec5.transformation.transz)

# Generation of heliostats with facets
H0 = Heliostat(SPX = 2, SPY = 2, HSX = 0.00945, HSY = 0.01284)
LOBJ5 = generateHfA(THEDEG=SZA, PHIDEG= PHI, PR=pRe, MINANG=140, MAXANG=220, GAPDEG = 10, FDRH=0.15,
                        NBH = 4, GAPDIST = 0.03, PILLH= 0.00517, REF=0.88, ROUGH = 0.001, HTYPE=H0)

# Without forgetting to add the reveicer at the list of objects
LOBJ5.append(Rec5)

### Drawing verification

In [ ]:
# visualize without facets
LOBJ5_bis = generateHfA(THEDEG=SZA, PHIDEG= PHI, PR=pRe, MINANG=140, MAXANG=220, GAPDEG = 10, FDRH=0.15,
                        NBH = 4, GAPDIST = 0.03, PILLH= 0.00517, HSX = 0.00945, HSY = 0.01284, REF=0.88, ROUGH = 0.001)
LOBJ5_bis.append(Rec5)
fig1 = visualize_entity(LOBJ5_bis, th_deg=SZA, ph_deg=PHI, show_rays=True)

### Run the simulation

In [ ]:
Pmin = [-0.6, -0.6, 0.]
Pmax = [0.6, 0.6, 0.125]
interval0 = [Pmin, Pmax] # enable to earn some computational time, can be very usefull in FF mode
aer = AerOPAC('desert', 0.5, 550.)
pro = AtmAFGL('afglms', comp=[aer])
lMode = CusForward(CFX = 8, CFY = 8, CFTX = -0.1, CFTY = 0., LMODE="FF", TYPE='isotropic', FOV=0.266)
m5 = Smartg(double = True, obj3D = True).run( surf = LambSurface(ALB=Albedo_cst(0.25)), 
        THVDEG=SZA, PHVDEG=PHI, NF=1e6, wl=550., NBPHOTONS=4e9, NBLOOP = 5e7, atm=pro,
        myObjects=LOBJ5, interval = interval0, cusL = lMode, DIRECT=True)
m5_ds = m5.to_xarray()


### Results

In [ ]:
m5.print_info(show_attrs=True)

In [ ]:
print("kernel time(s)=", m5.attrs['kernel time (s)'])
cat_view(m5_ds, output_unit='FLUX', flux_unit='kW')

In [ ]:
receiver_view(ds_sg_out=m5_ds, vmin=0, flux_unit='kW')

## The Same in backward

In [ ]:
# Specify the sun zenith angle and sun azimuth angle
SZA = 50
SAA = 100
PHI = 180. - SAA

# Creation of a receiver
wRx = 0.006
wRy = 0.007

Rec6 = Entity(name = "receiver", TC = 0.0005, \
                 materialAV = Matte(reflectivity = 0.), \
                 materialAR = Matte(reflectivity = 0.), \
                 geo = Plane( p1 = gc.Point(-wRx, -wRy, 0.),
                              p2 = gc.Point(wRx, -wRy, 0.),
                              p3 = gc.Point(-wRx, wRy, 0.),
                              p4 = gc.Point(wRx, wRy, 0.) ), \
                 transformation = Transformation( rotation = np.array([0., -101.5, 0.]), \
                                                  translation = np.array([0., 0., 0.1065]) ))

pRe = gc.Point(Rec6.transformation.transx, Rec6.transformation.transy, Rec6.transformation.transz)

# Generation of heliostats with facets
H0 = Heliostat(SPX = 2, SPY = 2, HSX = 0.00945, HSY = 0.01284)
LOBJ6 = generateHfA(THEDEG=SZA, PHIDEG= PHI, PR=pRe, MINANG=140, MAXANG=220, GAPDEG = 10, FDRH=0.15,
                        NBH = 4, GAPDIST = 0.03, PILLH= 0.00517, REF=0.88, ROUGH = 0.001, HTYPE=H0)

# Without forgetting to add the reveicer at the list of objects
LOBJ6.append(Rec6)

### Drawing verification

In [ ]:
# visualize without facets
LOBJ6_bis = generateHfA(THEDEG=SZA, PHIDEG= PHI, PR=pRe, MINANG=140, MAXANG=220, GAPDEG = 10, FDRH=0.15,
                        NBH = 4, GAPDIST = 0.03, PILLH= 0.00517, HSX = 0.00945, HSY = 0.01284, REF=0.88, ROUGH = 0.001)
LOBJ6_bis.append(Rec6)
fig1 = visualize_entity(LOBJ6_bis, th_deg=SZA, ph_deg=PHI)

### Run the simulation

In [ ]:
NormalRec = gc.Vector(0, 0, 1)
TT = gc.get_rotateY_tf(Rec6.transformation.rotation[1])
NormalRec = TT(NormalRec)
NormalRec = gc.normalize(NormalRec)

Pmin = [-0.6, -0.6, 0.]
Pmax = [0.6, 0.6, 0.03]
interval0 = [Pmin, Pmax] # enable to earn some computational time, can be very usefull in FF mode
aer = AerOPAC('desert', 0.5, 550.)
pro = AtmAFGL('afglms', comp=[aer])
lMode = CusBackward(POS=pRe, V=NormalRec, ALDEG=90, LMODE="BR", REC=Rec6)
m6 = Smartg(double = True, obj3D = True, back=True).run( surf = LambSurface(ALB=Albedo_cst(0.25)), 
        THVDEG=SZA, PHVDEG=PHI, NF=1e6, wl=550., NBPHOTONS=1e9, NBLOOP = 1e7, atm=pro,
        myObjects=LOBJ6, interval = interval0, cusL = lMode, SUN_DISC=0.266, DIRECT=True)
m6_ds = m6.to_xarray()


### Results

In [ ]:
print("kernel time(s)=", m6.attrs['kernel time (s)'])
cat_view(m6_ds, output_unit='FLUX', flux_unit='kW')

In [ ]:
receiver_view(ds_sg_out=m6_ds, vmin=0, flux_unit='kW')

# 2D Coastal Bathymetry — depth-varying ocean reflectance

This section demonstrates how ocean **depth** drives spectral reflectance changes along a coastal transect.

## Physical concept

In a shallow-water scenario:
- Photons enter the ocean at the surface, travel through the water column, reflect off the seafloor, and exit.
- The water column absorbs and scatters photons as a function of depth and wavelength.
- **Shallow water** → stronger seafloor contribution → higher reflectance, less spectrally selective.
- **Deep water** → seafloor signal attenuated exponentially → lower reflectance, selective (blue > red, because water absorbs red faster).

## Scene setup

```
x < 0 km : Land   (Lambertian, albedo = 0.10)
x > 0 km : Ocean  (FlatSurface + IOP_1, sandy seafloor albedo = 0.20)
```

The ocean depth increases linearly from the coast (`d_coast` at x=0) to `d_max` at x=5 km.
Because SmartG's ocean is 1D (single IOP per run), three separate runs are performed at
representative depths **d = 5 m**, **20 m**, and **100 m** and composited into a synthetic transect.

In [ ]:
# Additional imports needed for this section
import matplotlib.pyplot as plt
from smartg.smartg import RoughSurface, Environment
from smartg.albedo import Albedo_map
from smartg.water import IOP_1

In [ ]:
# ── Scene parameters ──────────────────────────────────────────────────────────
D_km   = 5.          # half-domain extent [km]
N      = 11          # number of sensors in the transect
depths = [5., 20., 100.]   # ocean depths to simulate [m]

# Wavelengths: three key ocean-colour bands
wl_bath = np.array([490., 550., 670.])   # [nm]

# ── Coastal environment (Albedo_map, ENV=5) ───────────────────────────────────
#   x < 0 : land  (environment index 0 → Albedo_cst(0.10))
#   x ≥ 0 : ocean (environment index -1 → uses surf + water from run())
#
# Albedo_map x-bins: (-inf, 0] = land, (0, +inf) = ocean
x_env = np.array([0., 1e8])   # [km] upper edges of x-bins
y_env = np.array([1e8])        # single y-bin (whole domain)
Ai_coast = np.array([[0],      # bin x<=0  : land (Alb_list index 0)
                     [-1]])    # bin x>0   : ocean (negative → surf+water)
Alb_land  = Albedo_cst(0.10)
ALB_coast = Albedo_map(Ai_coast, x_env, y_env, [Alb_land])
env_coast = Environment(ENV=5, X0=0., Y0=0., ALB=ALB_coast)

# ── Surface and atmosphere ────────────────────────────────────────────────────
surf_ocean = RoughSurface()                        # flat air/water interface
atm_bath   = AtmAFGL('afglt')                     # standard mid-latitude atmosphere

# ── Sensor transect: nadir-looking sensors from x=-4 km to x=+4 km ───────────
x_sensors = np.linspace(-D_km + 0.5, D_km - 0.5, N)   # [km]
sensors_bath = [Sensor(POSZ=120., POSX=xi, POSY=0., LOC='ATMOS', THDEG=180.)
                for xi in x_sensors]

# Sun zenith 30°, azimuth 0°
le_bath = {'th_deg': np.array([30.]), 'phi_deg': np.array([0.]), 'zip': True}

print(f"Transect x positions [km]: {x_sensors}")
print(f"Ocean depths to simulate [m]: {depths}")

In [ ]:
# ── Run one simulation per depth ─────────────────────────────────────────────
#
#  ocean model:  IOP_1(chl=0.1)  — coastal turbid water (chl = 0.1 mg/m³)
#  seafloor:     Albedo_cst(0.20) — sandy/coral bottom (20% reflectance)
#  ALB in IOP_1 is the *seafloor* albedo; setting DEPTH sets the water-column thickness.
#
#  Note: SmartG's ocean is 1-D (single global IOP per run), so we run once per
#  representative depth and then assemble a synthetic spatial transect.

res_bath = {}   # key = depth [m]

for d in depths:
    water_d = IOP_1(chl=0.1, DEPTH=d, ALB=Albedo_cst(0.20))
    m_d = Smartg().run(
        wl         = wl_bath,
        le         = le_bath,
        sensor     = sensors_bath,
        atm        = atm_bath,
        surf       = surf_ocean,
        env        = env_coast,
        water      = water_d,
        NBPHOTONS  = 5e7,
        BEER       = 1,
        RR         = 0,
    ).dropaxis('Zenith angles')
    res_bath[d] = m_d
    print(f"depth = {d:6.1f} m  →  mean ocean reflectance @ 490 nm: "
          f"{m_d['I_up (TOA)'].data[x_sensors > 0, 0].mean():.4f}")

### Visualise the depth-dependent reflectance

Expected physics:
- **Left plot**: land pixels (x < 0) are spectrally flat at ~0.10; ocean pixels decrease and become more blue-dominant as depth increases.
- **Right plot / depth curve**: at 670 nm (red, strong water absorption) the bottom signal vanishes quickly; at 490 nm (blue, low water absorption) the seafloor remains visible at greater depths.
- The coastline transition (x = 0) shows an abrupt change between land and ocean surface types.

In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────────
#
# Left panel  : reflectance transect (land → ocean) at three wavelengths, one
#               line per depth.
# Right panel : reflectance spectra at x=+2 km (ocean pixel) for each depth.

colors_d = {5.: '#1f77b4', 20.: '#ff7f0e', 100.: '#2ca02c'}
ls_wl    = {490.: '-',  550.: '--', 670.: ':'}
labels_wl = {490.: '490 nm', 550.: '550 nm', 670.: '670 nm'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── LEFT : spatial transect ───────────────────────────────────────────────────
ax = axes[0]
for d in depths:
    m = res_bath[d]
    R = m['I_up (TOA)'].data           # shape (N_sensors, N_wl)
    for i_w, wl_nm in enumerate(wl_bath):
        lbl = f"{int(wl_nm)} nm, d={int(d)} m" if wl_nm == 490. else None
        ax.plot(x_sensors, R[:, i_w],
                color=colors_d[d],
                ls=ls_wl[wl_nm],
                label=lbl)

ax.axvline(0, color='k', lw=0.8, ls='--', label='coastline')
ax.axvspan(-D_km, 0,  alpha=0.08, color='saddlebrown', label='land')
ax.axvspan(0, D_km,   alpha=0.08, color='steelblue',   label='ocean')
ax.set_xlabel('Cross-shore position [km]')
ax.set_ylabel('Reflectance  I/F  [–]')
ax.set_title('Reflectance transect (SZA=30°, nadir sensor)')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)

# ── RIGHT : depth vs reflectance spectra (ocean only) ────────────────────────
ax2 = axes[1]
# Pick the sensor closest to x=+2 km (ocean pixel)
i_oc = np.argmin(np.abs(x_sensors - 2.0))
for d in depths:
    m = res_bath[d]
    R_oc = m['I_up (TOA)'].data[i_oc, :]   # shape (N_wl,)
    ax2.plot(wl_bath, R_oc, 'o-', color=colors_d[d], label=f"depth = {int(d)} m")

ax2.set_xlabel('Wavelength [nm]')
ax2.set_ylabel('Reflectance  I/F  [–]')
ax2.set_title(f'Spectral signature at x≈{x_sensors[i_oc]:.1f} km (ocean pixel)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Depth–reflectance curves at fixed wavelengths ─────────────────────────────
fig2, ax3 = plt.subplots(figsize=(6, 4))
i_oc = np.argmin(np.abs(x_sensors - 2.0))
for i_w, wl_nm in enumerate(wl_bath):
    R_vs_d = [res_bath[d]['I_up (TOA)'].data[i_oc, i_w] for d in depths]
    ax3.plot(depths, R_vs_d, 'o-', label=labels_wl[wl_nm])

ax3.set_xscale('log')
ax3.set_xlabel('Ocean depth [m]')
ax3.set_ylabel('Reflectance  I/F  [–]')
ax3.set_title('Depth–reflectance relationship (coastal turbid water, seafloor alb = 0.20)')
ax3.legend()
ax3.grid(alpha=0.3)
plt.tight_layout()
plt.show()